# OPSD — Lightweight Evaluation on Google Colab

Short evaluation of your trained OPSD LoRA adapters **vs the base model**, reusing the repo's own eval method (`eval/evaluate_math.py`: vLLM + LoRA, `\boxed{}` answer extraction, `math_verify` grading, `Avg@N`). It auto-discovers any adapters saved under `opsd_outputs/` and prints a `Base vs trained` table — the same **format** as the tables in the project doc.

> **Honest scope.** This is a *lightweight* run designed to finish in ~15 min on **one** GPU, so it uses a **subset** of problems, a small `VAL_N`, a capped `MAX_NEW_TOKENS`, and non-thinking inference. The paper's headline numbers use full AIME, `val_n=12`, thinking mode, and up to 38912 tokens (≈30–50 min on 4×H100), so **absolute accuracies here will be lower and noisier** — especially for small models like Qwen3-0.6B, which is not one of the paper's models. Treat this as a *method/pipeline replication*, not a numeric reproduction.

> **Note on saved models.** A training run only yields a loadable adapter if it reached `save_steps` (or finished). Runs that stopped early (e.g. an A100 run that OOM'd before its first save) have **no weights** to evaluate and will be reported as such below.

## 1. GPU + Google Drive

Set the runtime to a GPU (**Runtime → Change runtime type → GPU**; an A100/L4 is ideal but a T4 also works for this lightweight run).

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type and select a GPU."
)
print("GPU:", torch.cuda.get_device_name(0))

# Mount Google Drive so we can read your trained adapters and write results.
from google.colab import drive
drive.mount("/content/drive")

## 2. Locate the project code

We need `eval/evaluate_math.py` from the OPSD project (the same folder you trained from). This finds it on your Drive automatically.

In [ ]:
import os, glob, sys

# Adjust if your OPSD folder lives elsewhere on Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/OPSD-main"

def find_project_dir(root="/content"):
    hits = glob.glob(os.path.join(root, "**", "opsd_train.py"), recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "opsd_train.py")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
else:
    PROJECT_DIR = find_project_dir()

assert PROJECT_DIR, "Could not locate the OPSD project (opsd_train.py). Put OPSD-main on your Drive."
EVAL_DIR = os.path.join(PROJECT_DIR, "eval")
assert os.path.exists(os.path.join(EVAL_DIR, "evaluate_math.py")), f"evaluate_math.py not found under {EVAL_DIR}"
print("PROJECT_DIR =", PROJECT_DIR)
print("EVAL_DIR    =", EVAL_DIR)

## 3. Install dependencies (vLLM eval stack)

Same pinned stack as training, plus `vllm` (the evaluator runs on vLLM with LoRA). vLLM is installed after the base stack, then `transformers`/`trl` are re-pinned so vLLM's resolver can't downgrade them.

> If Colab prompts to **restart the runtime** after install, do it, then re-run from cell 1. The `starlette`/`gradio` version warnings are harmless here.

In [ ]:
%pip install -q \
    "transformers==4.57.1" \
    "trl==0.26.0" \
    "peft==0.17.1" \
    "accelerate==1.11.0" \
    "datasets==3.6.0" \
    "math-verify==0.8.0" \
    "einops==0.8.1"

%pip install -q "vllm==0.11.0"
%pip install -q "transformers==4.57.1" "trl==0.26.0"

print("\nDone. If Colab asks you to restart the runtime, do it, then re-run from cell 1.")

## 4. Find your trained adapters

Scans `OUTPUT_DIR` for LoRA adapters (folders containing `adapter_model.safetensors`) and groups them by their base model (read from each `adapter_config.json`). Final adapters are used by default; set `INCLUDE_CHECKPOINTS = True` to also evaluate intermediate `checkpoint-*` dirs.

In [ ]:
import json
from collections import defaultdict

OUTPUT_DIR          = "/content/drive/MyDrive/opsd_outputs"  # where your training runs were saved
INCLUDE_CHECKPOINTS = False  # also evaluate intermediate checkpoint-* dirs?

def has_adapter(d):
    return (os.path.exists(os.path.join(d, "adapter_model.safetensors"))
            or os.path.exists(os.path.join(d, "adapter_model.bin")))

GROUPS = defaultdict(list)  # base_model -> [adapter_dir, ...]
found = []
for cfg in sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "adapter_config.json"), recursive=True)):
    d = os.path.dirname(cfg)
    if not has_adapter(d):
        continue
    if os.path.basename(d).startswith("checkpoint-") and not INCLUDE_CHECKPOINTS:
        continue
    with open(cfg) as f:
        base = json.load(f).get("base_model_name_or_path", "unknown")
    GROUPS[base].append(d)
    found.append((base, d))

print("=" * 70)
print("Discovered LoRA adapters to evaluate:")
if found:
    for base, d in found:
        print(f"  base={base}  ->  {os.path.relpath(d, OUTPUT_DIR)}")
else:
    print("  (none found — did training reach save_steps / finish?)")

# Flag run folders that produced no adapter (e.g. training stopped before save_steps).
for run in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    if os.path.isdir(run) and not has_adapter(run) and not any(d.startswith(run) for _, d in found):
        print(f"  [NO SAVED ADAPTER] {os.path.basename(run)}  (training likely stopped before save_steps)")
print("=" * 70)

## 5. Lightweight eval config

Defaults are tuned to fit ~15 min on one GPU. Knobs that trade speed for fidelity: `NUM_PROBLEMS`, `VAL_N`, `MAX_NEW_TOKENS`.

- `DATASET = "aime24"` matches the doc's headline benchmark, but it's hard — small models often score ~0%. For a clearer signal on Qwen3-0.6B, try `"math500"`.
- `ENABLE_THINKING = False` matches your non-thinking training and is much faster than thinking mode.

In [ ]:
DATASET         = "aime24"   # "aime24" | "aime25" | "hmmt25" | "math500" | "amc23" | "minerva"
NUM_PROBLEMS    = 15          # subset size (aime24 has 30). Lower = faster
VAL_N           = 2           # samples per problem (paper: 12). Lower = faster
MAX_NEW_TOKENS  = 2048        # generation cap (paper: up to 38912). Lower = faster
ENABLE_THINKING = False       # non-thinking matches your trained models + is far faster
TEMPERATURE     = 1.0         # doc eval temperature

print(f"dataset={DATASET}  problems={NUM_PROBLEMS}  val_n={VAL_N}  max_new_tokens={MAX_NEW_TOKENS}  thinking={ENABLE_THINKING}")
print(f"Total generations per model = {NUM_PROBLEMS} x {VAL_N} = {NUM_PROBLEMS * VAL_N}")

## 6. Run evaluation (base vs each adapter)

For every base model found, this evaluates the **base** once and then **each adapter** on top of it, calling the repo's `evaluate_math.py` (so grading matches the doc exactly). Output streams live; each model reloads vLLM (`enforce_eager=True`, so no CUDA-graph capture).

In [ ]:
import subprocess, shlex, time

RESULTS_DIR = os.path.join(OUTPUT_DIR, "eval_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = []  # filled below: one row per (base, model)

def _run_one(base_model, checkpoint_dir, out_json):
    cmd = f"""{sys.executable} -u evaluate_math.py \
      --base_model {base_model} \
      --dataset {DATASET} \
      --val_n {VAL_N} \
      --num_samples {NUM_PROBLEMS} \
      --max_new_tokens {MAX_NEW_TOKENS} \
      --max_model_len {MAX_NEW_TOKENS + 1024} \
      --temperature {TEMPERATURE} \
      --tensor_parallel_size 1 \
      --gpu_memory_utilization 0.85 \
      --output_file {shlex.quote(out_json)}"""
    if not ENABLE_THINKING:
        cmd += " --no_thinking"
    if checkpoint_dir is not None:
        cmd += f" --checkpoint_dir {shlex.quote(checkpoint_dir)}"
    env = dict(os.environ)
    env.update({"TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    proc = subprocess.Popen(shlex.split(cmd), cwd=EVAL_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0 or not os.path.exists(out_json):
        return None
    with open(out_json) as f:
        return json.load(f).get("average_at_n_pct")

if not GROUPS:
    print("No adapters discovered. Re-run cell 4 (check OUTPUT_DIR), then this cell.")

t0 = time.time()
for base, adapters in GROUPS.items():
    safe_base = base.replace("/", "_")
    print(f"\n{'#' * 70}\n# BASE MODEL: {base}\n{'#' * 70}")
    print(f"\n>>> [{DATASET}] base model (no adapter)")
    acc = _run_one(base, None, os.path.join(RESULTS_DIR, f"{safe_base}__base__{DATASET}.json"))
    RESULTS.append({"base": base, "model": "(base, no adapter)", "avg_at_n": acc})
    for d in adapters:
        rel = os.path.relpath(d, OUTPUT_DIR)
        print(f"\n>>> [{DATASET}] adapter: {rel}")
        tag = rel.replace("/", "_")
        acc = _run_one(base, d, os.path.join(RESULTS_DIR, f"{safe_base}__{tag}__{DATASET}.json"))
        RESULTS.append({"base": base, "model": rel, "avg_at_n": acc})

print(f"\nAll evaluations finished in {(time.time() - t0) / 60:.1f} min.")

## 7. Results table

`Base vs trained` comparison in the same shape as the project doc's tables. Re-runnable without re-evaluating (it just renders `RESULTS`).

In [ ]:
col = f"Avg@{VAL_N}"
print(f"Dataset: {DATASET}  |  problems: {NUM_PROBLEMS}  |  thinking: {ENABLE_THINKING}  |  max_new_tokens: {MAX_NEW_TOKENS}\n")
print(f"{'Base model':<22} {'Model':<42} {col:>10}")
print("-" * 76)
for r in RESULTS:
    acc = "n/a" if r["avg_at_n"] is None else f"{r['avg_at_n']:.1f}%"
    print(f"{r['base']:<22} {r['model']:<42} {acc:>10}")

# Save a compact summary next to the per-model JSONs.
summary_path = os.path.join(RESULTS_DIR, f"summary_{DATASET}_valn{VAL_N}.json")
with open(summary_path, "w") as f:
    json.dump({"dataset": DATASET, "num_problems": NUM_PROBLEMS, "val_n": VAL_N,
               "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": ENABLE_THINKING,
               "results": RESULTS}, f, indent=2)
print(f"\nSaved summary to: {summary_path}")

## Notes & next steps

- **To get numbers comparable to the doc:** evaluate a **Qwen3-1.7B** model trained for ~100 steps (with checkpoints saved), set `ENABLE_THINKING = True`, raise `VAL_N` toward 12, increase `MAX_NEW_TOKENS` (the paper uses up to 38912), and evaluate the full dataset (`NUM_PROBLEMS` = all). Expect ~30–50 min **per checkpoint** on strong GPUs.
- **Intermediate checkpoints:** set `INCLUDE_CHECKPOINTS = True` in cell 4 to plot accuracy vs training step (like the doc's per-step tables).
- **Missing adapters:** any run flagged `[NO SAVED ADAPTER]` never reached `save_steps`. Re-run that training so it saves (lower `save_steps`, ensure it doesn't OOM), then re-run this notebook — it will be picked up automatically.